# 🔟 Demo: PCA antes de K-Means con los dígitos de `load_digits`

> Material de respaldo de la sesión de PCA: **no se trabaja en clase**. Ejecútalo de arriba a abajo y lee las observaciones.

`load_digits` trae **1,797** dígitos escritos a mano en imágenes de 8 × 8 = **64 píxeles** (intensidad de 0 a 16). Conocemos el dígito real de cada imagen, así que podemos medir qué tan bien K-Means los agrupa **sin verlo**.

Pregunta: **¿mejora K-Means si primero reducimos las 64 dimensiones con PCA?**

Como aquí sí conocemos la respuesta real (el dígito), comparamos cada agrupamiento con ella mediante una **tabla cruzada** dígito × cluster. Para resumirla en un número contamos las **imágenes fuera del grupo de su dígito**: en cada cluster vemos cuál es el dígito más frecuente y contamos las imágenes que no son ese dígito (0 = cada cluster es un solo dígito). Este conteo sólo compara etiquetas, así que no depende del espacio. En la vida real no tendríamos la etiqueta.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

digits = load_digits()
X_digits = digits.data
y_digits = digits.target
print(f"{X_digits.shape[0]:,} imágenes × {X_digits.shape[1]} píxeles | rango {X_digits.min():.0f}-{X_digits.max():.0f}")

fig, axes = plt.subplots(1, 10, figsize=(10, 1.4))
for d, ax in enumerate(axes):
    ax.imshow(X_digits[y_digits == d][0].reshape(8, 8), cmap='binary')
    ax.set_title(str(d))
    ax.axis('off')
plt.show()


def fuera_de_su_digito(labels):
    """En cada cluster, cuenta las imágenes que NO son del dígito más frecuente de ese cluster."""
    tabla = pd.crosstab(y_digits, labels)
    return int((tabla.sum() - tabla.max()).sum())

---
## 1️⃣ Los dígitos en 2D

Primero, PCA para **ver**: proyectamos los 64 píxeles sobre PC1 y PC2 y coloreamos con el dígito real (sólo para validar).

In [ ]:
pca_2d = PCA(n_components=2, random_state=42)
scores_2d = pca_2d.fit_transform(X_digits)
print(f"Varianza explicada por PC1 + PC2: {pca_2d.explained_variance_ratio_.sum():.1%}")

fig = px.scatter(x=scores_2d[:, 0], y=scores_2d[:, 1], color=y_digits.astype(str),
                 category_orders={'color': [str(d) for d in range(10)]},
                 labels={'x': 'PC1', 'y': 'PC2', 'color': 'dígito'}, opacity=0.7,
                 title='load_digits proyectado en PC1-PC2 (color = dígito real)')
fig.update_layout(width=800, height=600)
fig.show()

✅ **Qué observar:** con sólo **28.5%** de la varianza algunos dígitos ya forman islas (el 0, el 6, el 4), pero otros se enciman en el centro (el 1, el 8, el 5). Dos dimensiones sirven para **ver**, no necesariamente para **agrupar**.

---
## 2️⃣ K-Means con distintos números de componentes

Ajustamos K-Means (k = 10) sobre los 64 píxeles y sobre los primeros $k$ componentes, y en cada caso contamos cuántas de las 1,797 imágenes quedan fuera del grupo de su dígito.

💡 No estandarizamos: los 64 píxeles están en la misma unidad (0 a 16). Lo comprobamos al final.

In [ ]:
filas = []
for n_comp in [2, 5, 10, 13, 20, 30, 64]:
    if n_comp < 64:
        pca_n = PCA(n_components=n_comp, random_state=42).fit(X_digits)
        Z = pca_n.transform(X_digits)
        varianza = pca_n.explained_variance_ratio_.sum()
    else:
        Z, varianza = X_digits, 1.0
    labels = KMeans(n_clusters=10, n_init=10, random_state=42).fit_predict(Z)
    filas.append({
        'dimensiones': n_comp,
        'varianza_conservada': round(varianza, 3),
        'fuera_de_su_digito': fuera_de_su_digito(labels)
    })
tabla_componentes = pd.DataFrame(filas)
tabla_componentes

In [ ]:
fig = px.line(tabla_componentes, x='dimensiones', y='fuera_de_su_digito', markers=True,
              labels={'fuera_de_su_digito': 'Imágenes fuera del grupo de su dígito'},
              title='¿Qué tan bien recupera K-Means los dígitos según las dimensiones?')
fig.update_layout(width=800, height=420)
fig.show()

✅ **Qué observar:**
- Con **2** componentes K-Means pierde mucho: **730 de 1,797** imágenes quedan fuera del grupo de su dígito. Demasiada información tirada
- A partir de unos **10** componentes el conteo se estabiliza en **≈ 370-385** imágenes, igual que con los 64 píxeles (**371**)
- 💡 PCA **no** hace que K-Means sea mejor aquí: da el **mismo** resultado con una fracción de las dimensiones. Eso ya vale la pena: menos memoria, menos tiempo y datos más fáciles de manejar

---
## 3️⃣ ⚠️ La trampa del silhouette

Una comparación muy común (y engañosa) es calcular el silhouette de K-Means en 64D, el silhouette de K-Means en el espacio PCA, y concluir "PCA mejoró el clustering X%". Veamos con el PCA que conserva el **80%** de la varianza.

In [ ]:
pca_80 = PCA(n_components=0.80, random_state=42).fit(X_digits)
X_pca80 = pca_80.transform(X_digits)
print(f"Componentes para el 80%: {pca_80.n_components_}")

kmeans_64 = KMeans(n_clusters=10, n_init=10, random_state=42).fit(X_digits)
kmeans_pca80 = KMeans(n_clusters=10, n_init=10, random_state=42).fit(X_pca80)

comparacion = pd.DataFrame([
    {
        'modelo': 'K-Means en 64 píxeles',
        'silhouette en su propio espacio': silhouette_score(X_digits, kmeans_64.labels_),
        'silhouette en 64D (misma referencia)': silhouette_score(X_digits, kmeans_64.labels_),
        'fuera_de_su_digito': fuera_de_su_digito(kmeans_64.labels_),
    },
    {
        'modelo': f'K-Means en {pca_80.n_components_} PCs (80%)',
        'silhouette en su propio espacio': silhouette_score(X_pca80, kmeans_pca80.labels_),
        'silhouette en 64D (misma referencia)': silhouette_score(X_digits, kmeans_pca80.labels_),
        'fuera_de_su_digito': fuera_de_su_digito(kmeans_pca80.labels_),
    },
]).round(3)
comparacion

In [ ]:
# Los dos agrupamientos entre sí: filas = cluster en 64D, columnas = cluster en PCA
tabla_64_vs_pca = pd.crosstab(kmeans_64.labels_, kmeans_pca80.labels_,
                              rownames=['cluster 64D'], colnames=['cluster PCA'])
# Para cada cluster de PCA, la celda más grande son las imágenes que coinciden con un cluster de 64D
print(f"Imágenes en el mismo grupo: {tabla_64_vs_pca.max().sum():,} de {len(y_digits):,}")
tabla_64_vs_pca

✅ **Qué observar:**
- Cada uno en **su propio espacio**, el silhouette "sube" de **0.182** a **0.240** (¡+31%!). Parece que PCA mejoró mucho
- Pero medidos en la **misma referencia** (64D), los dos agrupamientos tienen el mismo silhouette (**0.182**) y casi las mismas imágenes fuera del grupo de su dígito (**371** vs **383**)
- La tabla cruzada compara los dos agrupamientos entre sí: **1,745 de 1,797** imágenes (97%) quedan en el mismo grupo en ambos
- El silhouette subió porque en 13 dimensiones **todas** las distancias cambian, no porque los grupos sean mejores
- 💡 Regla: no compares silhouettes calculados en espacios distintos. Usa una referencia común o, si conoces la etiqueta real (datos de práctica), una tabla cruzada

---
## 4️⃣ Una excepción a "siempre escala"

En el curso repetimos "estandariza antes de K-Means y PCA". La razón es que las variables suelen estar en **unidades distintas**. Aquí todas son intensidades de 0 a 16. ¿Qué pasa si estandarizamos de todos modos?

In [ ]:
X_digits_scaled = StandardScaler().fit_transform(X_digits)
kmeans_scaled = KMeans(n_clusters=10, n_init=10, random_state=42).fit(X_digits_scaled)
print(f"Imágenes fuera del grupo de su dígito sin escalar: {fuera_de_su_digito(kmeans_64.labels_):,} de 1,797")
print(f"Imágenes fuera del grupo de su dígito escalando:   {fuera_de_su_digito(kmeans_scaled.labels_):,} de 1,797")
print(f"Píxeles casi constantes (desv. est. < 0.5): {(X_digits.std(axis=0) < 0.5).sum()} de 64")

✅ **Qué observar:** estandarizar **empeora** el resultado (de **371** a **571** imágenes fuera del grupo de su dígito). Hay píxeles de la orilla que casi siempre valen 0; al estandarizarlos, sus pocas variaciones (ruido) pesan lo mismo que los trazos del centro. Con variables en la **misma unidad**, piensa antes de escalar.

---
## 5️⃣ ¿Qué encontró K-Means?

Visualizamos los clusters de K-Means (sobre 13 PCs) en el plano PC1-PC2 y la "imagen promedio" de cada cluster.

In [ ]:
fig = px.scatter(x=scores_2d[:, 0], y=scores_2d[:, 1], color=kmeans_pca80.labels_.astype(str),
                 category_orders={'color': [str(c) for c in range(10)]},
                 labels={'x': 'PC1', 'y': 'PC2', 'color': 'cluster'}, opacity=0.7,
                 title=f'Clusters de K-Means ({pca_80.n_components_} PCs) vistos en PC1-PC2')
fig.update_layout(width=800, height=600)
fig.show()

# Imagen promedio de cada cluster (en píxeles originales) y el dígito más frecuente en él
fig, axes = plt.subplots(1, 10, figsize=(12, 1.8))
for c, ax in enumerate(axes):
    miembros = kmeans_pca80.labels_ == c
    ax.imshow(X_digits[miembros].mean(axis=0).reshape(8, 8), cmap='binary')
    moda = np.bincount(y_digits[miembros]).argmax()
    ax.set_title(f'C{c}: "{moda}"\n({(y_digits[miembros] == moda).mean():.0%})', fontsize=8)
    ax.axis('off')
plt.suptitle('Imagen promedio de cada cluster, su dígito más frecuente y qué % del cluster es ese dígito', y=1.12)
plt.show()

In [ ]:
# Tabla cruzada: filas = dígito real, columnas = cluster
pd.crosstab(y_digits, kmeans_pca80.labels_, rownames=['dígito'], colnames=['cluster'])

✅ **Qué observar:**
- La mayoría de los clusters tiene una imagen promedio que **parece un dígito** concreto, sin que K-Means haya visto las etiquetas
- En la tabla cruzada se ve dónde falla: los dígitos que se parecen (1, 8 y 9, por ejemplo) se reparten o se mezclan en los mismos clusters

## 📝 En resumen
1. PCA antes de K-Means da **el mismo resultado con menos dimensiones**; no es una mejora mágica de calidad
2. Para comparar agrupamientos en espacios distintos calcula la silhouette en **la misma referencia**; si tienes la etiqueta real, compárala con una **tabla cruzada**
3. Escalar es la regla cuando las unidades son distintas; con píxeles en la misma escala puede empeorar el resultado